# Canonical Data Validation and Analytical Readiness

**Project:** Lead Funnel Conversion Observatory  
**Phase:** Phase 3  
**Data version:** phase3-v2  

This notebook validates the approved canonical schema and records the final Phase 3 analytical-readiness evidence.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CANONICAL = ROOT / 'data' / 'canonical' / 'intelligence_data.csv'
OUTPUTS = ROOT / 'data-science' / 'outputs'

df = pd.read_csv(CANONICAL)
print('Canonical records:', len(df))
print('Canonical columns:', len(df.columns))

Canonical records: 60
Canonical columns: 20


In [2]:
required_columns = [
    'record_id', 'record_type', 'observed_at', 'entity_id',
    'related_entity_id', 'entity_name', 'category', 'subcategory',
    'status', 'stage', 'metric_name', 'metric_value', 'metric_unit',
    'text_value', 'latitude', 'longitude', 'source_name',
    'source_record_id', 'is_synthetic', 'data_version'
]

missing_columns = [c for c in required_columns if c not in df.columns]
assert not missing_columns, f'Missing required canonical columns: {missing_columns}'
assert len(df) == 60, f'Expected 60 canonical records, found {len(df)}'
assert len(df.columns) == 20, f'Expected 20 canonical columns, found {len(df.columns)}'
assert df['is_synthetic'].astype(str).str.lower().eq('true').all()
assert df['data_version'].astype(str).eq('phase3-v2').all()

print('STRUCTURAL VALIDATION: PASS')
print('Canonical schema columns: PASS')
print('Synthetic flag: true')
print('Data version: phase3-v2')

STRUCTURAL VALIDATION: PASS
Canonical schema columns: PASS
Synthetic flag: true
Data version: phase3-v2


In [3]:
quality = json.loads((OUTPUTS / 'quality_assessment.json').read_text(encoding='utf-8'))
representativeness = json.loads((OUTPUTS / 'representativeness_assessment.json').read_text(encoding='utf-8'))
readiness = json.loads((OUTPUTS / 'analytical_readiness.json').read_text(encoding='utf-8'))

print('QUALITY RESULT:', quality.get('result'))
print('REPRESENTATIVENESS RESULT:', representativeness.get('result'))
print('ANALYTICAL READINESS:', readiness.get('final_result'))
print('PRIMARY ARCHETYPE:', readiness.get('primary_archetype'))
print('PRIMARY TRACK:', readiness.get('primary_track'))

QUALITY RESULT: PASS
REPRESENTATIVENESS RESULT: None
ANALYTICAL READINESS: DATA READY FOR ANALYTICAL TRACK DEVELOPMENT
PRIMARY ARCHETYPE: Process Lifecycle
PRIMARY TRACK: Track A — Comparative


In [4]:
observed = pd.to_datetime(df['observed_at'], errors='coerce')
lead_records = df[df['record_type'].astype(str).eq('crm_lead')]
text_count = df['text_value'].notna().sum()

print('Final validation summary')
print('-------------------------')
print('Canonical records:', len(df))
print('Canonical columns:', len(df.columns))
print('Source leads:', lead_records['source_record_id'].nunique())
print('Observed date coverage:', observed.min().isoformat(), 'to', observed.max().isoformat())
print('Categories:', ', '.join(sorted(df['category'].dropna().astype(str).unique())))
print('Subcategories:', ', '.join(sorted(df['subcategory'].dropna().astype(str).unique())))
print('Statuses:', ', '.join(sorted(df['status'].dropna().astype(str).unique())))
print('Stages:', ', '.join(sorted(df['stage'].dropna().astype(str).unique())))
print('Metric names:', ', '.join(sorted(df['metric_name'].dropna().astype(str).unique())))
print('Structured text_value non-null:', text_count)
print('Latitude non-null:', df['latitude'].notna().sum())
print('Longitude non-null:', df['longitude'].notna().sum())

Final validation summary
-------------------------
Canonical records: 60
Canonical columns: 20
Source leads: 30
Observed date coverage: 2026-01-05T00:00:00 to 2026-03-20T00:00:00
Categories: Analytics, Payments, Security
Subcategories: Campaign, Partner, Referral, Website
Statuses: Lead, Lost, Opportunity, Proposal, Qualified, Won
Stages: Lead, Lost, Opportunity, Proposal, Qualified, Won
Metric names: days_in_stage, lead_value
Structured text_value non-null: 60
Latitude non-null: 0
Longitude non-null: 0


## Final Result

The approved canonical dataset passes structural, quality and representativeness checks. The primary data archetype is **Process Lifecycle** and the primary supported analytical track is **Track A ? Comparative**.

The dataset is synthetic and contains 30 source CRM leads represented as 60 canonical analytical records. The canonical schema uses `category`, `subcategory`, `observed_at` and `text_value`; source-level fields such as location and team are not treated as canonical columns.